# Chapter 2 — Tools and `tool_use` · Homework (self-test)

**Goal:** check that you can design tools and schemas, choose the right `tool_choice`, separate syntax from semantic errors, write a correct tool loop, and handle client and server tools in one harness.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 60–90 minutes
- Estimated API cost: **$0** (everything graded runs offline against fake responses); the one optional live cell costs < $0.01
- Prerequisites: the practice notebook, and the venv from [00-prerequisites](../../00-prerequisites/README.md)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 10 exam-style questions in the `answers` cell, then run the grader.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design answer, then compare with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises), the same bar as the Claude exam's 720/1000.

In [ ]:
import hashlib
import json
import os
import re
from dataclasses import dataclass, field
from typing import Any, Callable, Literal, Optional

import anthropic
from dotenv import find_dotenv, load_dotenv
from pydantic import BaseModel

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
client = anthropic.Anthropic()  # only the optional live exercise uses it
print("anthropic", anthropic.__version__, "| MODEL =", MODEL)

The optional live cell needs a key. Skip this assert if you have none: everything graded is offline.

In [ ]:
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env (see 00-prerequisites)"

### Helpers (run, don't edit)

`FakeBlock` / `FakeResponse` have the same attribute names as SDK responses (`.type`, `.text`, `.id`, `.name`, `.input`, `.stop_reason`), and `FakeClient` mimics `client.messages.create(...)`, so code that passes these checks also works with the real SDK.

In [ ]:
# ---------- grading bookkeeping ----------
RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _run_check(n: int, check: Callable[[], None]) -> None:
    """Run one checker; never let it crash the notebook."""
    try:
        check()
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    except NotImplementedError:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    except AssertionError as exc:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    except Exception as exc:  # noqa: BLE001 - report, don't crash
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {exc}")


# ---------- fake Messages API objects (same attribute names as the SDK) ----------
@dataclass
class FakeBlock:
    type: str
    text: Optional[str] = None
    id: Optional[str] = None
    name: Optional[str] = None
    input: Optional[dict] = None
    toolset_name: Optional[str] = None   # set on computer/browser toolset member calls
    tool_use_id: Optional[str] = None    # set on server tool result blocks
    content: Any = None                  # set on server tool result blocks


@dataclass
class FakeResponse:
    content: list
    stop_reason: str


def fake_text(t: str) -> FakeBlock:
    return FakeBlock("text", text=t)


def fake_tool(name: str, tool_input: dict, id: str, toolset_name: Optional[str] = None) -> FakeBlock:
    return FakeBlock("tool_use", id=id, name=name, input=tool_input, toolset_name=toolset_name)


def fake_server_call(name: str, tool_input: dict, id: str) -> FakeBlock:
    return FakeBlock("server_tool_use", id=id, name=name, input=tool_input)


def fake_search_result(tool_use_id: str, urls: list[str]) -> FakeBlock:
    return FakeBlock("web_search_tool_result", tool_use_id=tool_use_id,
                     content=[{"type": "web_search_result", "url": u} for u in urls])


class FakeClient:
    """Mimics client.messages.create(): returns scripted responses, records every call."""

    def __init__(self, *responses: FakeResponse):
        self.responses = list(responses)
        self.calls: list[dict] = []
        self.messages = self  # so that fake.messages.create(...) works

    def create(self, **kwargs) -> FakeResponse:
        self.calls.append({**kwargs, "messages": list(kwargs.get("messages", []))})  # snapshot
        if not self.responses:
            raise RuntimeError("FakeClient ran out of scripted responses: your loop called the API too often")
        return self.responses.pop(0)


# ---------- a tiny backend for the exercises ----------
CUSTOMERS = {48213: {"customer_id": 48213, "name": "Jane Doe", "email": "jane@example.com", "order_ids": ["ORD-5001"]}}
ORDERS = {"ORD-5001": {"order_id": "ORD-5001", "customer_id": 48213, "status": "delivered", "total_cents": 4999}}

print("helpers ready")

## Part A — Quiz (10 questions)

Pick one letter per question. Domains: tool design, `tool_choice`, structured output, the agent loop.

**Q1. (Customer support agent)** Compliance requires that refunds above $500 never execute without a manager's approval. Your agent calls `process_refund` through a client-side tool loop. Which design meets the requirement?

- A) Add a system-prompt rule: "Never refund more than $500 without manager approval."
- B) Put the rule in `process_refund`'s description and mark the tool `strict: true`.
- C) In the code that executes `process_refund`, check the amount; above $500, queue the refund for approval instead of paying and return that status as the `tool_result`.
- D) Set `tool_choice` to `{"type": "none"}` on turns where the user mentions an amount over $500.

**Q2. (Agent loop)** A response has `stop_reason: "tool_use"` and content `[text, tool_use(get_customer, id=t1), tool_use(lookup_order, id=t2)]`. Both tools succeed. Which next request is correct?

- A) Append an assistant message with only the text block, then a user message with both `tool_result` blocks.
- B) Append the full assistant content, then ONE user message whose content is `[tool_result t1, tool_result t2]` (any text only after them).
- C) Append the full assistant content, then two user messages, one `tool_result` each, so Claude processes them separately.
- D) Append the full assistant content, then a user message that starts with a short text summary followed by the two `tool_result` blocks.

**Q3. (Agent loop)** A response ends with `stop_reason: "max_tokens"`. Its last block is a `tool_use` for `process_refund` whose input looks incomplete. What should the loop do?

- A) Execute `process_refund` with the fields that are present; `strict: true` guarantees the input is valid.
- B) Send a `tool_result` with `is_error: true` saying the input was truncated, and continue the loop.
- C) Treat it like `end_turn` and show the text so far to the customer.
- D) Do not execute the call; retry the request with a higher `max_tokens` (or escalate).

**Q4. (Structured data extraction (current API))** A pipeline on `claude-opus-5-5` must return a JSON object of contract metadata on every call. No action is taken; the JSON is the product. Which approach fits best?

- A) `client.messages.parse(output_format=ContractMetadata)` (or `output_config.format` with a JSON schema), checking `stop_reason` before trusting the result.
- B) Define an `extract_metadata` tool and set `tool_choice` to `{"type": "tool", "name": "extract_metadata"}`.
- C) Tell the model "respond only with JSON" and extract the first `{...}` with a regex.
- D) Set `tool_choice` to `{"type": "any"}` and mark the extraction tool `strict: true`.

**Q5. (Strict tool use)** You enabled `strict: true` on all tools. In a normal response (`stop_reason: "tool_use"`), which problem can STILL happen?

- A) `customer_id` arrives as the string `"48213"` although the schema says integer.
- B) The required `order_id` key is missing from the input.
- C) A well-formed `customer_id` that belongs to no real customer.
- D) The `tool_use` block names a tool that is not in your `tools` list.

**Q6. (Structured data extraction)** About 35% of contracts state no renewal date. Your schema has `renewal_date` as a required string, and reviewers keep finding plausible but invented dates. What is the best fix?

- A) Keep it required and add "do not invent dates" to the system prompt.
- B) Keep the key required but make it nullable (`["string", "null"]`), describe it as "null if the contract states no renewal date", and accept `null` as a valid result.
- C) Retry the extraction until the date passes a calendar validity check.
- D) Add a `YYYY-MM-DD` regex `pattern` to the schema so only real dates can be produced.

**Q7. (Structured data extraction)** A ticket classifier uses `category` enum `["billing", "shipping", "technical"]`. Analysts find that data-deletion requests and partnership inquiries are all labeled `technical`, and that information is lost. Which schema change fixes this?

- A) Replace the enum with a free-text `category` string.
- B) Add many few-shot examples showing how to map unusual tickets into the three categories.
- C) Make `category` optional so the model can leave it out when unsure.
- D) Add `"other"` (and `"unclear"`) to the enum plus a nullable `category_detail` string that explains other/unclear cases.

**Q8. (Agent loop)** Which termination logic is correct for a manual client-side tool loop?

- A) Continue while `stop_reason == "tool_use"`; return on `end_turn`; handle `max_tokens` and `refusal` explicitly; keep a `max_turns` cap as a safety net.
- B) Stop when the assistant's text contains "done" or "finished".
- C) Always run exactly 3 iterations, which is enough for most tasks.
- D) Stop as soon as a response contains any text block.

**Q9. (Tool error handling)** `lookup_order` gets a 404 from the backend because the customer typed `ORD-50O1` (letter O instead of zero). What should your code send back?

- A) Nothing for this call: send the next request without its `tool_result`; Claude will notice it is missing.
- B) A `tool_result` with content `"Operation failed"` and no `is_error` flag.
- C) A `tool_result` with `is_error: true` and content like "Order ORD-50O1 not found. Order IDs are ORD- followed by 4 digits; ask the customer to re-check it."
- D) Raise an exception that ends the conversation so a human can take over.

**Q10. (Tool design)** An agent has 18 tools, including `search_orders` ("Searches orders") and `find_order` ("Finds an order"). About 30% of calls go to the wrong tool. What is the most effective FIRST step?

- A) Put a routing classifier in front of the agent to pick the tool.
- B) Rewrite the descriptions (purpose, inputs, what each returns, when not to use it), remove the overlap between `search_orders` and `find_order` by re-scoping or renaming one, and give the agent only the tools its role needs.
- C) Add 30 few-shot examples of correct tool calls to the system prompt.
- D) Set `tool_choice` to `{"type": "any"}` so that a tool is always called.

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
    "q9": None,
    "q10": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e', 'q2': '61e5302385b2dd8204dbddfd86b357cacb0de07dbc800460aa335d9bc96b6ca1', 'q3': '14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6', 'q4': '0eca085218afe99ae6fb326d50c6e730cd71c102ccfe2f9a10d8e6054e2018e0', 'q5': 'e3c6d5ddd445920945724cd165c914deaf97d5041f947ba2df35e569269b353f', 'q6': '621f4cab9e29d186b9dabeec79a433d2126c30a9e3f5712a1411fc67d73f1e40', 'q7': '45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457', 'q8': 'f98cc196593da1da1a7b4550be81b590cd44e7098497c2034a83c78f8a98f3bc', 'q9': 'e19608e84d37f3c94b5f88738c75a8c96772a5e61784e3e24b5512fa65ff63c7', 'q10': 'c69a9867de9de4cc2e45f4cd6a6128833af714d8993edbeac622795c9c5e20d3'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker.

### Exercise 1 — Write a strict tool definition

Write `build_lookup_order_tool()` returning a tool definition dict for **`lookup_order`** that:

- sets `strict: True` (top level of the tool) and uses an `object` schema with `additionalProperties: False`;
- has a required string `order_id` and an **optional** boolean `include_items`;
- uses no keywords strict mode doesn't support (`minLength`, `maximum`, ...);
- has a description of **at least 3 sentences** that says what it returns, and when **not** to use it, pointing to `get_customer` for finding customers.

Theory: [2.2](README.md#22-tool-definition), [2.4](README.md#24-json-schemas-for-structured-output)

In [ ]:
def build_lookup_order_tool() -> dict:
    """Return a strict tool definition for lookup_order (see the task above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    def check():
        tool = build_lookup_order_tool()
        assert isinstance(tool, dict), "return a dict"
        assert tool.get("name") == "lookup_order", "name must be 'lookup_order'"
        assert tool.get("strict") is True, "set strict: True at the top level of the tool (not inside input_schema)"
        schema = tool.get("input_schema", {})
        assert schema.get("type") == "object", "input_schema.type must be 'object'"
        assert schema.get("additionalProperties") is False, "strict schemas need additionalProperties: False"
        props = schema.get("properties", {})
        assert props.get("order_id", {}).get("type") == "string", "order_id must be a string property"
        assert "order_id" in schema.get("required", []), "order_id must be required"
        assert props.get("include_items", {}).get("type") == "boolean", "include_items must be a boolean property"
        assert "include_items" not in schema.get("required", []), "include_items is optional: leave it out of required"
        banned = {"minLength", "maxLength", "minimum", "maximum", "multipleOf", "maxItems"}
        dumped = json.dumps(schema)
        used = [k for k in banned if f'"{k}"' in dumped]
        assert not used, f"strict schemas don't support {used}; validate those rules in code"
        desc = tool.get("description", "")
        sentences = [s for s in re.split(r"(?<=[.!?])\s+", desc.strip()) if s]
        assert len(sentences) >= 3, f"description has {len(sentences)} sentence(s); write at least 3"
        assert "get_customer" in desc, "description should say when to use get_customer instead (a boundary)"
        assert re.search(r"\b(do not|don't|not for|instead)\b", desc, re.I), "say when NOT to use this tool"
    _run_check(1, check)


check_ex1()

### Exercise 2 — Pick the right `tool_choice`

Write `tool_choice_for(model, must_call=None, single_call=False, forbid=False)`:

- `forbid=True` → `{"type": "none"}`.
- `must_call="x"` on a model that supports forced tool use → `{"type": "tool", "name": "x"}`.
- `must_call` on a model in `REJECTS_FORCED` (Opus 5.5, Sonnet 5.5, Fable 5.1, Mythos 5.1) → `{"type": "auto"}` (you will steer with the prompt and verify in code).
- otherwise → `{"type": "auto"}`.
- `single_call=True` adds `"disable_parallel_tool_use": True` **inside** the returned dict.

Theory: [2.3](README.md#23-the-tool_choice-parameter)

In [ ]:
REJECTS_FORCED = {"claude-opus-5-5", "claude-sonnet-5-5", "claude-fable-5-1", "claude-mythos-5-1"}


def tool_choice_for(model: str, must_call: str | None = None, single_call: bool = False,
                    forbid: bool = False) -> dict:
    """Return the tool_choice dict for this request (rules in the task above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    def check():
        cases = [
            (("claude-haiku-4-5",), {}, {"type": "auto"}),
            (("claude-haiku-4-5",), {"must_call": "extract_metadata"}, {"type": "tool", "name": "extract_metadata"}),
            (("claude-sonnet-5-5",), {"must_call": "extract_metadata"}, {"type": "auto"}),
            (("claude-opus-5-5",), {"must_call": "extract_metadata", "single_call": True},
             {"type": "auto", "disable_parallel_tool_use": True}),
            (("claude-opus-5",), {"must_call": "x", "single_call": True},
             {"type": "tool", "name": "x", "disable_parallel_tool_use": True}),
            (("claude-sonnet-5-5",), {"forbid": True}, {"type": "none"}),
            (("claude-haiku-4-5",), {"single_call": True}, {"type": "auto", "disable_parallel_tool_use": True}),
        ]
        for args, kwargs, expected in cases:
            got = tool_choice_for(*args, **kwargs)
            assert got == expected, f"tool_choice_for({args[0]!r}, **{kwargs}) returned {got}, expected {expected}"
    _run_check(2, check)


check_ex2()

### Exercise 3 — Design an honest extraction schema (Pydantic)

Define `ReceiptExtraction` for `client.messages.parse(output_format=ReceiptExtraction)`:

- `merchant` (string, always present), `total_cents` (integer);
- `receipt_date`: **required but nullable** (the key is always there, `null` when the receipt shows no date);
- `payment_method`: one of `card`, `cash`, `mobile`, plus `other` and `unclear`;
- `payment_detail`: required + nullable string explaining `other` / `unclear`.

The checker inspects `anthropic.transform_schema(ReceiptExtraction)`, which is exactly what `messages.parse()` sends.

Theory: [2.4](README.md#24-json-schemas-for-structured-output)

In [ ]:
class ReceiptExtraction(BaseModel):
    # TODO: merchant, receipt_date, total_cents, payment_method, payment_detail
    pass

In [ ]:
def _allows_null(prop: dict) -> bool:
    types = prop.get("type")
    return ("null" in types if isinstance(types, list) else types == "null") or any(
        _allows_null(v) for v in prop.get("anyOf", []))


def check_ex3() -> None:
    def check():
        schema = anthropic.transform_schema(ReceiptExtraction)  # what messages.parse() would send
        props, required = schema.get("properties", {}), set(schema.get("required", []))
        fields = ["merchant", "receipt_date", "total_cents", "payment_method", "payment_detail"]
        missing = [f for f in fields if f not in props]
        assert not missing, f"missing fields: {missing}"
        not_required = [f for f in fields if f not in required]
        assert not not_required, (f"{not_required} are not required. Use Optional[str] WITHOUT a default "
                                  "(required + nullable), not Optional[str] = None")
        for f in ("receipt_date", "payment_detail"):
            assert _allows_null(props[f]), f"{f} must be nullable (Optional[str])"
        for f in ("merchant", "total_cents", "payment_method"):
            assert not _allows_null(props[f]), f"{f} should not be nullable"
        assert props["total_cents"].get("type") == "integer", "total_cents must be an int (cents, no float money)"
        enum = props["payment_method"].get("enum", [])
        for v in ("card", "cash", "mobile", "other", "unclear"):
            assert v in enum, f"payment_method enum must include {v!r} (got {enum})"
        sample = ReceiptExtraction.model_validate_json(json.dumps({
            "merchant": "Cafe Uno", "receipt_date": None, "total_cents": 1250,
            "payment_method": "other", "payment_detail": "voucher"}))
        assert sample.receipt_date is None
    _run_check(3, check)


check_ex3()

### Exercise 4 — Catch semantic errors a schema can't

Write `invoice_semantic_errors(inv)` for a dict shaped like
`{"line_items": [{"description", "amount_cents"}], "discount_cents", "stated_total_cents", "issue_date", "due_date"}` (dates are ISO strings or `None`). Return a list of messages, `[]` if consistent:

- sum of line amounts minus `discount_cents` ≠ `stated_total_cents` → one message containing the word **total**;
- `due_date` earlier than `issue_date` (both present) → one message containing **due_date**;
- any line amount ≤ 0 → a message containing **line**;
- `None` dates are **not** errors (the data just isn't on the document).

Theory: [2.5](README.md#25-syntax-vs-semantic-errors)

In [ ]:
def invoice_semantic_errors(inv: dict) -> list[str]:
    """Return a list of human-readable problems; [] when the invoice is consistent."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    def check():
        base = {"line_items": [{"description": "Widget", "amount_cents": 4000},
                               {"description": "Shipping", "amount_cents": 750}],
                "discount_cents": 500, "stated_total_cents": 4250,
                "issue_date": "2026-09-01", "due_date": "2026-10-01"}
        got = invoice_semantic_errors(base)
        assert got == [], f"a consistent invoice should give [], got {got}"
        got = invoice_semantic_errors({**base, "stated_total_cents": 4750})
        assert len(got) == 1 and "total" in got[0].lower(), f"total mismatch: expected 1 error mentioning 'total', got {got}"
        got = invoice_semantic_errors({**base, "due_date": "2026-08-15"})
        assert len(got) == 1 and "due_date" in got[0], f"due before issue: expected 1 error mentioning 'due_date', got {got}"
        got = invoice_semantic_errors({**base, "issue_date": None, "due_date": None})
        assert got == [], f"null dates mean 'not on the document', which is not an error; got {got}"
        bad_line = {**base, "line_items": [{"description": "Widget", "amount_cents": 4750},
                                           {"description": "Credit", "amount_cents": -500}],
                    "discount_cents": 0}
        got = invoice_semantic_errors(bad_line)
        assert any("line" in e.lower() for e in got), f"negative line amount should give an error mentioning 'line', got {got}"
    _run_check(4, check)


check_ex4()

### Exercise 5 — Build the `tool_result` message

Write `build_tool_results(response, handlers)`. For **every** `tool_use` block in `response.content` (skip text), in order:

- run `handlers[block.name](**block.input)` and add `{"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(output)}`;
- if the handler raises, add a result with `"is_error": True` and the exception message in `content`;
- if the tool name is unknown, add an `is_error: True` result whose content names the tool.

Return **one** user message: `{"role": "user", "content": [...]}` with only `tool_result` blocks.

Theory: [2.6](README.md#26-the-client-side-tool-loop-and-tool_result-rules-expanded-beyond-the-guide)

In [ ]:
def build_tool_results(response, handlers: dict[str, Callable[..., Any]]) -> dict:
    """Execute every tool_use block in response.content and return ONE user message of tool_result blocks."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    def check():
        def get_customer(customer_id):
            if not isinstance(customer_id, int):
                raise ValueError("customer_id must be numeric, e.g. 48213")
            return CUSTOMERS[customer_id]

        response = FakeResponse(stop_reason="tool_use", content=[
            fake_text("Let me look that up."),
            fake_tool("get_customer", {"customer_id": 48213}, "t1"),
            fake_tool("get_customer", {"customer_id": "jane"}, "t2"),
            fake_tool("delete_everything", {}, "t3"),
        ])
        msg = build_tool_results(response, {"get_customer": get_customer})
        assert isinstance(msg, dict) and msg.get("role") == "user", "return {'role': 'user', 'content': [...]}"
        content = msg.get("content")
        assert isinstance(content, list) and all(isinstance(b, dict) for b in content), "content must be a list of dicts"
        assert all(b.get("type") == "tool_result" for b in content), "content must contain ONLY tool_result blocks (no text)"
        ids = [b.get("tool_use_id") for b in content]
        assert ids == ["t1", "t2", "t3"], f"one tool_result per tool_use, same order; got ids {ids}"
        ok, bad_input, unknown = content
        assert not ok.get("is_error"), "a successful call must not set is_error"
        assert json.loads(ok["content"]) == CUSTOMERS[48213], "successful content should be json.dumps(handler output)"
        assert bad_input.get("is_error") is True, "a handler exception must become is_error: True (never drop it)"
        assert "customer_id must be numeric" in bad_input.get("content", ""), "include the exception message so the model can fix its call"
        assert unknown.get("is_error") is True and "delete_everything" in unknown.get("content", ""), \
            "an unknown tool must return is_error: True naming the tool"
    _run_check(5, check)


check_ex5()

### Exercise 6 — Implement the loop control flow

Write `run_support_loop(llm, user_text, tools, handlers, max_turns=5)`. Call `llm.messages.create(model=MODEL, max_tokens=1024, tools=tools, messages=messages)` each turn, append the **full** `response.content` as the assistant message, then branch on `stop_reason`:

| `stop_reason` | Do | Return `status` |
|---|---|---|
| `tool_use` | run the tools, append **one** user message with all results, loop | (keep going) |
| `end_turn` | stop; `text` = all text blocks joined | `"done"` |
| `max_tokens` | stop, **do not** run tools | `"truncated"` |
| `refusal` | stop, do not run tools | `"refused"` |
| cap reached | at most `max_turns` API calls | `"max_turns"` |

Return `{"status": ..., "text": ..., "messages": messages}`. The checker drives your loop with `FakeClient`, so no API calls happen.

Theory: [2.6](README.md#26-the-client-side-tool-loop-and-tool_result-rules-expanded-beyond-the-guide)

In [ ]:
def run_support_loop(llm, user_text: str, tools: list[dict], handlers: dict[str, Callable[..., Any]],
                     max_turns: int = 5) -> dict:
    """Manual tool loop. Return {"status": ..., "text": ..., "messages": [...]} (rules in the task above)."""
    # TODO (you may reuse build_tool_results from Exercise 5)
    raise NotImplementedError

In [ ]:
def check_ex6() -> None:
    def check():
        calls = []

        def lookup_order(order_id):
            calls.append(order_id)
            return ORDERS.get(order_id, {"error": "not found"})

        handlers, tools = {"lookup_order": lookup_order}, [{"name": "lookup_order", "input_schema": {"type": "object"}}]

        # 1) parallel tool calls, then end_turn
        first = FakeResponse(stop_reason="tool_use", content=[
            fake_text("Checking both."), fake_tool("lookup_order", {"order_id": "ORD-5001"}, "a"),
            fake_tool("lookup_order", {"order_id": "ORD-9"}, "b")])
        fake = FakeClient(first, FakeResponse(stop_reason="end_turn", content=[fake_text("All done.")]))
        out = run_support_loop(fake, "status of ORD-5001 and ORD-9?", tools, handlers)
        assert isinstance(out, dict), "return a dict with status/text/messages"
        assert out.get("status") == "done", f"end_turn should give status 'done', got {out.get('status')!r}"
        assert out.get("text") == "All done.", f"text should be the final text blocks joined, got {out.get('text')!r}"
        assert len(fake.calls) == 2, f"expected 2 API calls, got {len(fake.calls)}"
        assert all(c.get("tools") == tools for c in fake.calls), "pass tools= on every request"
        sent = fake.calls[1]["messages"]
        assert [m["role"] for m in sent] == ["user", "assistant", "user"], \
            f"2nd request roles should be user/assistant/user, got {[m['role'] for m in sent]}"
        assert list(sent[1]["content"]) == first.content, "append the FULL response.content (text + tool_use), not just text"
        res = sent[2]["content"]
        assert [b.get("tool_use_id") for b in res] == ["a", "b"], "both results, in ONE user message, matched by id"
        assert calls == ["ORD-5001", "ORD-9"], f"each tool should run exactly once; handler calls: {calls}"

        # 2) truncated turn: do NOT run the (incomplete) tool call
        calls.clear()
        fake = FakeClient(FakeResponse(stop_reason="max_tokens",
                                       content=[fake_tool("lookup_order", {"order_id": "ORD-50"}, "c")]))
        out = run_support_loop(fake, "x", tools, handlers)
        assert out.get("status") == "truncated", f"max_tokens should give 'truncated', got {out.get('status')!r}"
        assert calls == [], "never execute tool calls from a max_tokens turn"

        # 3) refusal
        fake = FakeClient(FakeResponse(stop_reason="refusal", content=[]))
        assert run_support_loop(fake, "x", tools, handlers).get("status") == "refused", "refusal should give 'refused'"

        # 4) a model that never stops calling tools
        endless = [FakeResponse(stop_reason="tool_use", content=[fake_tool("lookup_order", {"order_id": "ORD-5001"}, f"t{i}")])
                   for i in range(10)]
        fake = FakeClient(*endless)
        out = run_support_loop(fake, "x", tools, handlers, max_turns=3)
        assert out.get("status") == "max_turns", f"hitting the cap should give 'max_turns', got {out.get('status')!r}"
        assert len(fake.calls) == 3, f"max_turns=3 must mean at most 3 API calls, got {len(fake.calls)}"
    _run_check(6, check)


check_ex6()

### Exercise 7 — Plan the next step for a mixed client/server turn

Your harness has client tools **and** server tools (`web_search`, `web_fetch`) in one `tools` array. Write `plan_next_step(response, handlers)`, where `handlers` maps **(`toolset_name` or `""`, `name`)** to a function. Return `{"next": ..., "message": ..., "pending_server_ids": [...]}`:

| `stop_reason` | `next` | `message` | `pending_server_ids` |
|---|---|---|---|
| `pause_turn` | `"resend"` (send the assistant content back as-is) | `None` | `[]` |
| `tool_use` | `"send_results"` | one user message with a `tool_result` per **client** `tool_use` block, in order | ids of `server_tool_use` blocks that have **no** result block in this response |
| anything else | `"stop"` | `None` | `[]` |

Rules for the results:
- **never** answer a `server_tool_use` block (the API runs server tools);
- dispatch on the (`toolset_name`, `name`) pair, so a computer-use `screenshot` and a custom `screenshot` don't collide;
- a toolset member's result echoes `"toolset_name"`; other results don't have that key;
- unknown tools and handler exceptions become `is_error: True` results.

Server result blocks have a `type` ending in `_tool_result` and a `tool_use_id`. The checker uses `FakeBlock`s, so no API calls happen.

Theory: [2.7](README.md#27-client-tools-anthropic-schema-tools-and-server-tools-beyond-the-guide)

In [ ]:
def plan_next_step(response, handlers: dict[tuple[str, str], Callable[..., Any]]) -> dict:
    """Decide what the harness does after this response (rules in the task above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex7() -> None:
    def check():
        ran = []
        handlers = {
            ("", "save_note"): lambda text: ran.append(("save_note", text)) or "saved",
            ("computer", "screenshot"): lambda: ran.append(("computer", "screenshot")) or "image",
            ("", "screenshot"): lambda: ran.append(("custom", "screenshot")) or "WRONG HANDLER",
        }
        mixed = FakeResponse(stop_reason="tool_use", content=[
            fake_text("Searching and saving."),
            fake_server_call("web_search", {"query": "q1"}, "srvtoolu_1"),
            fake_search_result("srvtoolu_1", ["https://a.example"]),
            fake_server_call("web_fetch", {"url": "https://b.example"}, "srvtoolu_2"),   # no result yet
            fake_tool("save_note", {"text": "hi"}, "t1"),
            fake_tool("screenshot", {}, "t2", toolset_name="computer"),
        ])
        out = plan_next_step(mixed, handlers)
        assert isinstance(out, dict) and out.get("next") == "send_results", \
            f"a tool_use stop should give next='send_results', got {out!r}"
        msg = out.get("message") or {}
        assert msg.get("role") == "user", "message must be {'role': 'user', 'content': [...]}"
        blocks = msg.get("content") or []
        ids = [b.get("tool_use_id") for b in blocks]
        assert "srvtoolu_1" not in ids and "srvtoolu_2" not in ids, \
            "never send a tool_result for a server_tool_use block: the API runs server tools itself"
        assert ids == ["t1", "t2"], f"one tool_result per CLIENT tool_use, in order; got {ids}"
        assert all(b.get("type") == "tool_result" for b in blocks), "content must be only tool_result blocks"
        assert ("custom", "screenshot") not in ran and ("computer", "screenshot") in ran, \
            "dispatch on the (toolset_name, name) pair: the computer screenshot must not hit the custom handler"
        assert blocks[1].get("toolset_name") == "computer", "a toolset member's result must echo toolset_name"
        assert "toolset_name" not in blocks[0], "only toolset member results carry toolset_name"
        assert out.get("pending_server_ids") == ["srvtoolu_2"], \
            f"pending_server_ids should list server calls with no result block: expected ['srvtoolu_2'], got {out.get('pending_server_ids')}"

        unknown = FakeResponse(stop_reason="tool_use", content=[fake_tool("drop_tables", {}, "t9")])
        b = plan_next_step(unknown, handlers)["message"]["content"][0]
        assert b.get("is_error") is True and b.get("tool_use_id") == "t9", "an unknown tool must get an is_error result"

        paused = FakeResponse(stop_reason="pause_turn", content=[fake_server_call("web_search", {"query": "q"}, "s9")])
        out = plan_next_step(paused, handlers)
        assert out.get("next") == "resend" and out.get("message") is None, \
            "pause_turn: resend the assistant content as-is (next='resend', message=None), no tool_result"
        done = FakeResponse(stop_reason="end_turn", content=[fake_text("bye")])
        assert plan_next_step(done, handlers).get("next") == "stop", "end_turn should give next='stop'"
    _run_check(7, check)


check_ex7()

### Exercise 6b — your loop against the real API (live, optional, ~$0.01, not scored)

Run it once Exercise 6 passes. Same code, real model, one strict tool.

In [ ]:
# (live, optional, ~$0.01, not scored) your loop against the real API with a strict tool
live_tools = [{
    "name": "lookup_order", "strict": True,
    "description": ("Fetch one order by its order ID (format ORD-####). Returns status and total in cents. "
                    "Use it for questions about an order's status or price. Do not use it to find customers."),
    "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}},
                     "required": ["order_id"], "additionalProperties": False},
}]
if not RESULTS.get(6):
    print("Finish Exercise 6 first (its checker must pass), then run this cell.")
else:
    out = run_support_loop(client, "What's the status of ORD-5001?", live_tools,
                           {"lookup_order": lambda order_id: ORDERS.get(order_id, {"error": "not found"})})
    print(out["status"], "|", out["text"])

## Part C — Architecture scenario

**Insurance claims intake agent.** An insurer wants an agent (on `claude-sonnet-5-5`) that reads incoming claim emails with photos and PDFs, extracts the claim data, checks it against the policy system, and **pays claims up to $1,000 automatically**; larger or doubtful claims go to a human adjuster. Volume: ~3,000 claims/day. Regulators require an audit trail of every payout decision.

Describe your design (10–20 lines):
1. The tool set (names, one-line descriptions, which ones are strict) and how you keep tool selection reliable.
2. How you get the claim data out of the documents (which structured-output mechanism, key schema decisions).
3. Where each guarantee comes from: model vs. API vs. your code (especially the $1,000 rule).
4. Error handling: transient backend failures, invalid inputs, missing information.
5. The loop: termination, limits, and what happens on `max_tokens` / `refusal`.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **Small, role-scoped toolset** with non-overlapping names and rich descriptions, e.g. `get_policy(policy_id)`, `get_claim_history(policy_id)`, `submit_payout(claim_id, amount_cents, ...)`, `escalate_to_adjuster(claim_id, reason)`. All tools `strict: true` with `additionalProperties: false`.
- **Extraction with structured outputs** (`messages.parse()` + Pydantic or `output_config.format`), not a forced extraction tool (Sonnet 5.5 rejects forced `tool_choice`). Required + **nullable** fields for data that may be absent (incident date, police report number), enums with `"other"` / `"unclear"`, amounts in integer cents, case-insensitive enum handling.
- **Semantic validation in code**: policy active on the incident date, claimed amount ≤ coverage, line items add up, no duplicate claim. Retry with the specific error only when the answer is in the documents; otherwise `null` and escalate.
- **The $1,000 rule enforced in code** inside `submit_payout` (and ideally a precondition such as a validated-claim token from a preceding check), never only in the prompt or description. Above the limit, or any doubt, the tool queues for an adjuster.
- **Errors**: timeouts retried with backoff inside the tool; validation errors returned immediately as `is_error: true` with actionable text; every `tool_use` gets a `tool_result`.
- **Loop**: driven by `stop_reason`, full `response.content` appended, parallel results in one message, `max_turns` cap; `max_tokens` → retry with more room, never run truncated calls; `refusal` → route to a human.
- **Audit log** of every tool call, input, result and decision, written by the code (not by the model). Untrusted email content stays inside the user/tool_result content, never in the system prompt.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_ex = 7
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (10 + n_ex)
print(f"Quiz:       {quiz}/10 ({quiz / 10:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: on to Chapter 3" if overall >= 0.72 else "RETRY: reread the README sections behind your ✗ items, then try again")

Next chapter: [Chapter 3 — Agent SDK](../03-agent-sdk/README.md)

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **C.** Claude only *requests* calls; your code executes them, so a hard business rule belongs in that code. A and B are probabilistic (a prompt or description can be ignored; `strict` only guarantees shape). D depends on text detection and blocks legitimate work instead of routing it to approval.

**Q2:** **B.** Every `tool_use` needs a matching `tool_result` in the user message right after it, all parallel results go in one message, and `tool_result` blocks come first. A drops the `tool_use` blocks (400). C splits results (400 for the missing id, and it teaches Claude to stop making parallel calls). D puts text before the results (400).

**Q3:** **D.** A `max_tokens` stop means the output was cut off, so the tool input may be incomplete; strict mode's guarantee explicitly excludes truncated replies. Never run a truncated irreversible call (A). B and C hide the real problem instead of re-running the turn with more room.

**Q4:** **A.** Structured outputs are the purpose-built way to get schema-valid JSON and work on Opus 5.5. B and D are forced tool use, which Opus 5.5 rejects with a 400. C gives up every guarantee.

**Q5:** **C.** Strict mode is grammar-constrained: types (A), required keys (B) and tool names (D) are guaranteed. It guarantees shape, not truth: a perfectly formatted but non-existent ID is a semantic error your code must catch.

**Q6:** **B.** Required + nullable gives the model an honest "not present" answer, so it stops fabricating. A is a weak prompt patch. C can't create information that isn't in the source. D constrains format, not truth: an invented date in the right format still passes.

**Q7:** **D.** `"other"` + a detail field keeps unanticipated information while preserving a closed, aggregatable set, and `"unclear"` gives an honest low-confidence signal. A loses the structure analysts need. B still forces everything into three buckets. C hides the problem.

**Q8:** **A.** `stop_reason` is the structured control signal; `max_turns` is only a safety net. B is fragile text parsing, C cuts off longer tasks and wastes calls on short ones, and D is wrong because Claude often writes text *and* a `tool_use` in the same turn.

**Q9:** **C.** Return the failure as `is_error: true` with an actionable message so Claude can recover (here, ask the customer to re-check the ID). A breaks the protocol (400). B is generic and unflagged. D throws away a recoverable situation.

**Q10:** **B.** Descriptions are the model's routing signal; fixing vague, overlapping descriptions and trimming the toolset fixes the root cause cheaply. A and C are heavier layers on a broken interface. D forces *a* call but not the *right* one (and is rejected on Opus 5.5 / Sonnet 5.5).
</details>

<details><summary>Exercise 1 — Write a strict tool definition: reference solution</summary>

```python
def build_lookup_order_tool() -> dict:
    return {
        "name": "lookup_order",
        "description": (
            "Fetch one order by its order ID (format ORD-#### such as ORD-5001). "
            "Returns status, total in cents and, if include_items is true, the line items. "
            "Use it for questions about an order's status, contents or price. "
            "Do not use it to identify a customer; call get_customer for that."
        ),
        "strict": True,
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order ID, e.g. ORD-5001"},
                "include_items": {"type": "boolean", "description": "Also return line items. Default false."},
            },
            "required": ["order_id"],
            "additionalProperties": False,
        },
    }
```
</details>

<details><summary>Exercise 2 — Pick the right `tool_choice`: reference solution</summary>

```python
REJECTS_FORCED = {"claude-opus-5-5", "claude-sonnet-5-5", "claude-fable-5-1", "claude-mythos-5-1"}


def tool_choice_for(model: str, must_call: str | None = None, single_call: bool = False,
                    forbid: bool = False) -> dict:
    if forbid:
        return {"type": "none"}
    if must_call and model not in REJECTS_FORCED:
        choice = {"type": "tool", "name": must_call}
    else:
        choice = {"type": "auto"}  # forced use is rejected here: steer with the prompt + verify in code
    if single_call:
        choice["disable_parallel_tool_use"] = True
    return choice
```
</details>

<details><summary>Exercise 3 — Design an honest extraction schema (Pydantic): reference solution</summary>

```python
class ReceiptExtraction(BaseModel):
    merchant: str
    receipt_date: Optional[str]        # required key, may be null (no "= None" default!)
    total_cents: int
    payment_method: Literal["card", "cash", "mobile", "other", "unclear"]
    payment_detail: Optional[str]      # explanation for other/unclear, else null
```
</details>

<details><summary>Exercise 4 — Catch semantic errors a schema can't: reference solution</summary>

```python
from datetime import date


def invoice_semantic_errors(inv: dict) -> list[str]:
    errors = []
    lines = inv["line_items"]
    if any(item["amount_cents"] <= 0 for item in lines):
        errors.append("Every line amount must be positive; put discounts in discount_cents.")
    expected = sum(item["amount_cents"] for item in lines) - inv["discount_cents"]
    if expected != inv["stated_total_cents"]:
        errors.append(f"Lines minus discount = {expected} but stated_total_cents = {inv['stated_total_cents']}.")
    issue, due = inv.get("issue_date"), inv.get("due_date")
    if issue and due and date.fromisoformat(due) < date.fromisoformat(issue):
        errors.append(f"due_date {due} is before issue_date {issue}.")
    return errors
```
</details>

<details><summary>Exercise 5 — Build the `tool_result` message: reference solution</summary>

```python
def build_tool_results(response, handlers: dict[str, Callable[..., Any]]) -> dict:
    results = []
    for block in response.content:
        if block.type != "tool_use":
            continue
        if block.name not in handlers:
            results.append({"type": "tool_result", "tool_use_id": block.id, "is_error": True,
                            "content": f"Unknown tool '{block.name}'. Use one of: {sorted(handlers)}."})
            continue
        try:
            output = handlers[block.name](**block.input)
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(output)})
        except Exception as exc:  # noqa: BLE001 - report every failure to the model
            results.append({"type": "tool_result", "tool_use_id": block.id, "is_error": True,
                            "content": f"{type(exc).__name__}: {exc}"})
    return {"role": "user", "content": results}
```
</details>

<details><summary>Exercise 6 — Implement the loop control flow: reference solution</summary>

```python
def run_support_loop(llm, user_text: str, tools: list[dict], handlers: dict[str, Callable[..., Any]],
                     max_turns: int = 5) -> dict:
    messages: list[dict] = [{"role": "user", "content": user_text}]
    for _ in range(max_turns):
        response = llm.messages.create(model=MODEL, max_tokens=1024, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": response.content})  # full content, always
        if response.stop_reason == "tool_use":
            messages.append(build_tool_results(response, handlers))
            continue
        if response.stop_reason == "end_turn":
            text = "".join(b.text for b in response.content if b.type == "text")
            return {"status": "done", "text": text, "messages": messages}
        status = {"max_tokens": "truncated", "refusal": "refused"}.get(response.stop_reason, response.stop_reason)
        return {"status": status, "text": "", "messages": messages}  # never run tools from these turns
    return {"status": "max_turns", "text": "", "messages": messages}
```
</details>

<details><summary>Exercise 7 — Plan the next step for a mixed client/server turn: reference solution</summary>

```python
def plan_next_step(response, handlers: dict[tuple[str, str], Callable[..., Any]]) -> dict:
    if response.stop_reason == "pause_turn":
        return {"next": "resend", "message": None, "pending_server_ids": []}
    if response.stop_reason != "tool_use":
        return {"next": "stop", "message": None, "pending_server_ids": []}
    answered = {b.tool_use_id for b in response.content if b.type.endswith("_tool_result")}
    pending = [b.id for b in response.content if b.type == "server_tool_use" and b.id not in answered]
    results = []
    for b in response.content:
        if b.type != "tool_use":                      # never answer server_tool_use blocks
            continue
        key = (b.toolset_name or "", b.name)
        res = {"type": "tool_result", "tool_use_id": b.id}
        if b.toolset_name:
            res["toolset_name"] = b.toolset_name
        if key not in handlers:
            results.append({**res, "is_error": True, "content": f"Unknown tool {key}"})
            continue
        try:
            results.append({**res, "content": handlers[key](**b.input)})
        except Exception as exc:  # noqa: BLE001
            results.append({**res, "is_error": True, "content": f"{type(exc).__name__}: {exc}"})
    return {"next": "send_results", "message": {"role": "user", "content": results}, "pending_server_ids": pending}
```
</details>